# Bab 3: Distribusi Probabilitas

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab03/beta_stunting.py": "from scipy.stats import beta\n\n# Prior dari angka nasional (ilustrasi): rerata 0,20, \"setara\" 20 balita\na0, b0 = 4, 16                                   # Beta(4, 16)  #<1>\nprint(f\"prior : rerata {a0/(a0+b0):.3f}\")\n\nfor nama, n, m in [(\"Desa A\", 12, 5), (\"Desa B\", 120, 50)]:\n    p_ml = m / n                                 # maximum likelihood  #<2>\n    a, b = a0 + m, b0 + n - m                    # posterior konjugat  #<3>\n    lo, hi = beta.ppf([0.025, 0.975], a, b)      # selang kredibel 95%\n    print(f\"{nama}: {m:2d}/{n:3d} | ML {p_ml:.3f} | \"\n          f\"posterior {a/(a+b):.3f}, 95% [{lo:.3f}, {hi:.3f}]\")\n",
"bab03/bias_varians.py": "import numpy as np\n\nrng = np.random.default_rng(1)\nsigma2 = 3.5**2                                  # varians sebenarnya: 12.25\nfor N in (2, 3, 5, 10, 50):\n    X = rng.normal(85.0, 3.5, size=(200_000, N)) # 200 ribu \"posyandu\"  #<1>\n    s2_ml = X.var(axis=1)                        # bagi N (ddof=0)  #<2>\n    s2_tb = X.var(axis=1, ddof=1)                # bagi N-1 (tak bias)\n    print(f\"N={N:2d} | E[s2_ML] = {s2_ml.mean():5.2f} \"\n          f\"(teori {(N-1)/N*sigma2:5.2f}) | E[s2_N-1] = {s2_tb.mean():5.2f}\")\n",
"bab03/gauss_sekuensial.py": "import numpy as np\n\n# Ilustrasi: rerata tinggi anak usia 24 bulan di satu kecamatan\nrng = np.random.default_rng(3)\nsigma = 3.5                          # simpangan baku diketahui (cm)\nx = rng.normal(84.0, sigma, size=50) # \"data posyandu\", rerata sebenarnya 84\n\nm, s2 = 87.0, 2.0**2                 # prior N(87, 2^2) dari kurva rujukan  #<1>\nfor n, xn in enumerate(x, start=1):\n    s2_baru = 1 / (1/s2 + 1/sigma**2)            # presisi bertambah  #<2>\n    m = s2_baru * (m/s2 + xn/sigma**2)           # rerata tertimbang presisi\n    s2 = s2_baru\n    if n in (1, 2, 5, 10, 50):\n        print(f\"N={n:2d} | ML = {x[:n].mean():6.2f} | \"\n              f\"posterior = {m:6.2f} +- {np.sqrt(s2):.2f}\")\n\n# Sekuensial = sekaligus: rumus batch untuk N = 50  #<3>\nN = len(x)\ns2_N = 1 / (1/2.0**2 + N/sigma**2)\nm_N = s2_N * (87.0/2.0**2 + x.sum()/sigma**2)\nprint(f\"batch N=50          posterior = {m_N:6.2f} +- {np.sqrt(s2_N):.2f}\")\n",
"bab03/gaussian_bersyarat.py": "import numpy as np\n\n# Ilustrasi: tinggi (cm) dan berat (kg) balita usia sekitar 2 tahun\nmu = np.array([85.0, 11.5])\ns_t, s_b, rho = 3.5, 1.3, 0.7\nSigma = np.array([[s_t**2, rho*s_t*s_b], [rho*s_t*s_b, s_b**2]])  #<1>\n\n# Distribusi berat bersyarat tinggi = 80 cm (rumus tertutup)  #<2>\nt = 80.0\nmu_bs = mu[1] + Sigma[1, 0] / Sigma[0, 0] * (t - mu[0])\nvar_bs = Sigma[1, 1] - Sigma[1, 0]**2 / Sigma[0, 0]\nprint(f\"marginal berat      : {mu[1]:.2f} +- {np.sqrt(Sigma[1,1]):.2f} kg\")\nprint(f\"berat | tinggi=80cm : {mu_bs:.2f} +- {np.sqrt(var_bs):.2f} kg (rumus)\")\n\n# Periksa dengan simulasi: ambil sampel yang tingginya dekat 80 cm  #<3>\nrng = np.random.default_rng(0)\nX = rng.multivariate_normal(mu, Sigma, size=2_000_000)\ndekat = np.abs(X[:, 0] - t) < 0.1\nprint(f\"berat | tinggi~80cm : {X[dekat,1].mean():.2f} +- {X[dekat,1].std():.2f} kg \"\n      f\"(simulasi, {dekat.sum()} sampel)\")\n",
"bab03/kde_tinggi.py": "import numpy as np\nfrom scipy.stats import norm\n\n# Ilustrasi: tinggi anak 24 bulan, campuran 25% stunting dan 75% normal\nrng = np.random.default_rng(7)\ndef sampel(n):\n    stunting = rng.random(n) < 0.25\n    return np.where(stunting, rng.normal(80, 3, n), rng.normal(87, 3.2, n))\nx_latih, x_val = sampel(200), sampel(1000)\n\ndef kde(x, data, h):                                  #<1>\n    return norm.pdf((x[:, None] - data[None, :]) / h).mean(axis=1) / h\n\nfor h in (0.3, 0.8, 1.5, 3.0, 6.0):\n    ll = np.log(kde(x_val, x_latih, h)).mean()        # log-likelihood validasi  #<2>\n    print(f\"h = {h:3.1f} cm      | log-likelihood validasi rata-rata = {ll:.3f}\")\n\nh_silverman = 1.06 * x_latih.std() * len(x_latih) ** (-1/5)   #<3>\nprint(f\"aturan Silverman  | h = {h_silverman:.2f} cm\")\n\ng = norm(x_latih.mean(), x_latih.std())               # pembanding: satu Gaussian\nprint(f\"satu Gaussian     | log-likelihood validasi rata-rata = \"\n      f\"{g.logpdf(x_val).mean():.3f}\")\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab03')
print('folder kerja:', os.getcwd())

## Listing 3.1: Prevalensi stunting dua desa: maximum likelihood dan posterior beta

Berkas: `kode/bab03/beta_stunting.py`

In [ ]:
from scipy.stats import beta

# Prior dari angka nasional (ilustrasi): rerata 0,20, "setara" 20 balita
a0, b0 = 4, 16                                   # Beta(4, 16)  # (1)
print(f"prior : rerata {a0/(a0+b0):.3f}")

for nama, n, m in [("Desa A", 12, 5), ("Desa B", 120, 50)]:
    p_ml = m / n                                 # maximum likelihood  # (2)
    a, b = a0 + m, b0 + n - m                    # posterior konjugat  # (3)
    lo, hi = beta.ppf([0.025, 0.975], a, b)      # selang kredibel 95%
    print(f"{nama}: {m:2d}/{n:3d} | ML {p_ml:.3f} | "
          f"posterior {a/(a+b):.3f}, 95% [{lo:.3f}, {hi:.3f}]")

## Listing 3.2: Berat badan bersyarat tinggi badan pada Gaussian dua dimensi

Berkas: `kode/bab03/gaussian_bersyarat.py`

In [ ]:
import numpy as np

# Ilustrasi: tinggi (cm) dan berat (kg) balita usia sekitar 2 tahun
mu = np.array([85.0, 11.5])
s_t, s_b, rho = 3.5, 1.3, 0.7
Sigma = np.array([[s_t**2, rho*s_t*s_b], [rho*s_t*s_b, s_b**2]])  # (1)

# Distribusi berat bersyarat tinggi = 80 cm (rumus tertutup)  # (2)
t = 80.0
mu_bs = mu[1] + Sigma[1, 0] / Sigma[0, 0] * (t - mu[0])
var_bs = Sigma[1, 1] - Sigma[1, 0]**2 / Sigma[0, 0]
print(f"marginal berat      : {mu[1]:.2f} +- {np.sqrt(Sigma[1,1]):.2f} kg")
print(f"berat | tinggi=80cm : {mu_bs:.2f} +- {np.sqrt(var_bs):.2f} kg (rumus)")

# Periksa dengan simulasi: ambil sampel yang tingginya dekat 80 cm  # (3)
rng = np.random.default_rng(0)
X = rng.multivariate_normal(mu, Sigma, size=2_000_000)
dekat = np.abs(X[:, 0] - t) < 0.1
print(f"berat | tinggi~80cm : {X[dekat,1].mean():.2f} +- {X[dekat,1].std():.2f} kg "
      f"(simulasi, {dekat.sum()} sampel)")

## Listing 3.3: Memeriksa bias varians maximum likelihood dengan simulasi

Berkas: `kode/bab03/bias_varians.py`

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
sigma2 = 3.5**2                                  # varians sebenarnya: 12.25
for N in (2, 3, 5, 10, 50):
    X = rng.normal(85.0, 3.5, size=(200_000, N)) # 200 ribu "posyandu"  # (1)
    s2_ml = X.var(axis=1)                        # bagi N (ddof=0)  # (2)
    s2_tb = X.var(axis=1, ddof=1)                # bagi N-1 (tak bias)
    print(f"N={N:2d} | E[s2_ML] = {s2_ml.mean():5.2f} "
          f"(teori {(N-1)/N*sigma2:5.2f}) | E[s2_N-1] = {s2_tb.mean():5.2f}")

## Listing 3.4: Pembaruan Bayesian sekuensial untuk rerata tinggi badan

Berkas: `kode/bab03/gauss_sekuensial.py`

In [ ]:
import numpy as np

# Ilustrasi: rerata tinggi anak usia 24 bulan di satu kecamatan
rng = np.random.default_rng(3)
sigma = 3.5                          # simpangan baku diketahui (cm)
x = rng.normal(84.0, sigma, size=50) # "data posyandu", rerata sebenarnya 84

m, s2 = 87.0, 2.0**2                 # prior N(87, 2^2) dari kurva rujukan  # (1)
for n, xn in enumerate(x, start=1):
    s2_baru = 1 / (1/s2 + 1/sigma**2)            # presisi bertambah  # (2)
    m = s2_baru * (m/s2 + xn/sigma**2)           # rerata tertimbang presisi
    s2 = s2_baru
    if n in (1, 2, 5, 10, 50):
        print(f"N={n:2d} | ML = {x[:n].mean():6.2f} | "
              f"posterior = {m:6.2f} +- {np.sqrt(s2):.2f}")

# Sekuensial = sekaligus: rumus batch untuk N = 50  # (3)
N = len(x)
s2_N = 1 / (1/2.0**2 + N/sigma**2)
m_N = s2_N * (87.0/2.0**2 + x.sum()/sigma**2)
print(f"batch N=50          posterior = {m_N:6.2f} +- {np.sqrt(s2_N):.2f}")

## Listing 3.5: Estimasi kerapatan kernel untuk tinggi badan, dengan $h$ dipilih dengan data validasi

Berkas: `kode/bab03/kde_tinggi.py`

In [ ]:
import numpy as np
from scipy.stats import norm

# Ilustrasi: tinggi anak 24 bulan, campuran 25% stunting dan 75% normal
rng = np.random.default_rng(7)
def sampel(n):
    stunting = rng.random(n) < 0.25
    return np.where(stunting, rng.normal(80, 3, n), rng.normal(87, 3.2, n))
x_latih, x_val = sampel(200), sampel(1000)

def kde(x, data, h):                                  # (1)
    return norm.pdf((x[:, None] - data[None, :]) / h).mean(axis=1) / h

for h in (0.3, 0.8, 1.5, 3.0, 6.0):
    ll = np.log(kde(x_val, x_latih, h)).mean()        # log-likelihood validasi  # (2)
    print(f"h = {h:3.1f} cm      | log-likelihood validasi rata-rata = {ll:.3f}")

h_silverman = 1.06 * x_latih.std() * len(x_latih) ** (-1/5)   # (3)
print(f"aturan Silverman  | h = {h_silverman:.2f} cm")

g = norm(x_latih.mean(), x_latih.std())               # pembanding: satu Gaussian
print(f"satu Gaussian     | log-likelihood validasi rata-rata = "
      f"{g.logpdf(x_val).mean():.3f}")